In [ ]:
from pathlib import Path
import os
import random

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "creditcard_2023.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "creditcard_2023.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(RESULTS_DIR / ".matplotlib"))
os.environ.setdefault("KERAS_HOME", str(RESULTS_DIR / ".keras"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
df = pd.read_csv(DATA_PATH)

In [ ]:
df.head()

Features do nosso dataset possui seguintes descrições:
id: Identificador único de cada transação
V1-V28: Features anonimizadas representando vários atributos de transação (e.g., hora, local, etc.)
Amount: Quantia da transação
Class: Label binário indicando se a transação é fraudulenta (1) ou não (0)

### Primeiramente, limpar os dados retirando informações com "NA" e ajustando variáveis de treinamento e teste

In [ ]:
df_new = df.dropna()

In [ ]:
df_new = df_new.drop(columns=['id'])

In [ ]:
df_new.head()

### Agora separar as features do target

In [ ]:
X = df_new.drop(columns=['Class'])
y = df_new['Class']




### separando treino, validação e teste:




In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

### Posteriormente, normalizamos os valores de Amount

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### Aqui uma diferença bem interessante de Autoencoders para Regressão Logística vem do fato de filtrar o treino usando somente as transações **normais**.

In [ ]:
X_train_normal = X_train_scaled[y_train == 0]
X_val_normal = X_val_scaled[y_val == 0]

In [ ]:
input_dim = X_train_normal.shape[1]

autoencoder = keras.Sequential([
    keras.Input(shape=(input_dim,)),

    # Encoder
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),

    # Decoder
    layers.Dense(16, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(input_dim, activation="linear")
])

autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

In [ ]:
history = autoencoder.fit(
    X_train_normal,
    X_train_normal,
    epochs=50,
    batch_size=256,
    validation_data=(X_val_normal, X_val_normal),
    shuffle=True
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Autoencoder Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "autoencoder_training_loss.png", dpi=300, bbox_inches="tight")
plt.show()

### Próxima etapa é fazer o cálculo de errp de reconstrução

In [ ]:
import numpy as np

X_val_pred = autoencoder.predict(X_val_scaled)

reconstruction_error_val = np.mean(
    np.square(X_val_scaled - X_val_pred),
    axis=1
)

### Definição de threshold. Aqui, utilizaremos 99%

In [ ]:
reconstruction_error_val_normal = reconstruction_error_val[y_val == 0]

threshold = np.percentile(reconstruction_error_val_normal, 99)

print("Threshold:", threshold)

In [ ]:
X_test_pred = autoencoder.predict(X_test_scaled)

reconstruction_error_test = np.mean(
    np.square(X_test_scaled - X_test_pred),
    axis=1
)

y_test_pred = (reconstruction_error_test > threshold).astype(int)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
)

cm = confusion_matrix(y_test, y_test_pred)
report = classification_report(y_test, y_test_pred)

print(cm)
print(report)

with open(RESULTS_DIR / "autoencoder_classification_report.txt", "w", encoding="utf-8") as f:
    f.write(report)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Not Fraud", "Fraud"]
)
disp.plot()
plt.title("Confusion Matrix - Autoencoder")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix_autoencoder.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

roc_auc = roc_auc_score(y_test, reconstruction_error_test)
pr_auc = average_precision_score(y_test, reconstruction_error_test)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

autoencoder_metrics = pd.DataFrame({
    "Model": ["Autoencoder (99th percentile threshold)"],
    "Accuracy": [accuracy_score(y_test, y_test_pred)],
    "Precision": [precision_score(y_test, y_test_pred, zero_division=0)],
    "Recall": [recall_score(y_test, y_test_pred, zero_division=0)],
    "F1-score": [f1_score(y_test, y_test_pred, zero_division=0)],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc],
})

autoencoder_metrics.to_csv(RESULTS_DIR / "autoencoder_metrics.csv", index=False)
autoencoder_metrics

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(reconstruction_error_test[y_test == 0], bins=50, alpha=0.6, label="Normal")
plt.hist(reconstruction_error_test[y_test == 1], bins=50, alpha=0.6, label="Fraud")
plt.axvline(threshold, linestyle="--", label="Threshold")
plt.xlabel("Reconstruction Error")
plt.ylabel("Frequency")
plt.title("Reconstruction Error Distribution")
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "autoencoder_reconstruction_error_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
percentiles = [95, 97, 98, 99, 99.2, 99.5, 99.7, 99.8, 99.9]

results = []

for p in percentiles:
    threshold_candidate = np.percentile(reconstruction_error_val[y_val == 0], p)
    y_pred_temp = (reconstruction_error_test > threshold_candidate).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_temp).ravel()

    results.append({
        "percentile": p,
        "threshold": threshold_candidate,
        "accuracy": accuracy_score(y_test, y_pred_temp),
        "precision_fraud": precision_score(y_test, y_pred_temp, zero_division=0),
        "recall_fraud": recall_score(y_test, y_pred_temp, zero_division=0),
        "f1_fraud": f1_score(y_test, y_pred_temp, zero_division=0),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    })

threshold_results = pd.DataFrame(results)
threshold_results.to_csv(RESULTS_DIR / "autoencoder_threshold_results.csv", index=False)
threshold_results

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(threshold_results["percentile"], threshold_results["precision_fraud"], marker="o", label="Precision")
plt.plot(threshold_results["percentile"], threshold_results["recall_fraud"], marker="o", label="Recall")
plt.plot(threshold_results["percentile"], threshold_results["f1_fraud"], marker="o", label="F1-score")
plt.xlabel("Normal reconstruction-error percentile used as threshold")
plt.ylabel("Fraud-class score")
plt.title("Autoencoder Threshold Tuning")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "autoencoder_threshold_tuning.png", dpi=300, bbox_inches="tight")
plt.show()

### Comparação final dos modelos

In [ ]:
best_autoencoder = threshold_results.sort_values(
    ["f1_fraud", "recall_fraud"],
    ascending=False
).iloc[0]

supervised_metrics = pd.read_csv(RESULTS_DIR / "logistic_regression_metrics.csv")
unsupervised_metrics = pd.DataFrame({
    "Model": ["Autoencoder (best F1 threshold)"],
    "Accuracy": [best_autoencoder["accuracy"]],
    "Precision": [best_autoencoder["precision_fraud"]],
    "Recall": [best_autoencoder["recall_fraud"]],
    "F1-score": [best_autoencoder["f1_fraud"]],
    "ROC-AUC": [roc_auc],
    "PR-AUC": [pr_auc],
    "Best Threshold Percentile": [best_autoencoder["percentile"]],
    "Threshold": [best_autoencoder["threshold"]],
})

for col in ["Best Threshold Percentile", "Threshold"]:
    if col not in supervised_metrics.columns:
        supervised_metrics[col] = np.nan

model_comparison = pd.concat(
    [supervised_metrics, unsupervised_metrics],
    ignore_index=True,
    sort=False,
)
model_comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
model_comparison

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC", "PR-AUC"]
plot_df = model_comparison.set_index("Model")[metrics_to_plot]

ax = plot_df.T.plot(kind="bar", figsize=(10, 6))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Supervised vs Unsupervised Fraud Detection Results")
plt.xticks(rotation=35, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison_metrics.png", dpi=300, bbox_inches="tight")
plt.show()